# [Phase 1] D043 HWP 파일 파싱 및 JSON 변환
**대전대학교_2024학년도_다층적_융합_학습경험_플랫폼_MILE__전.hwp**

기존 파이프라인에서 파싱 실패한 파일 1건만 처리합니다.  
나머지 98개 파일은 전혀 건드리지 않습니다.

### 핵심 변경
`hwp5proc xml` (subprocess) 대신 **`hwp5.xmlmodel.Hwp5File` Python API**를 직접 호출합니다.  
subprocess는 별도 프로세스라 monkey-patch가 적용되지 않지만,  
Python API는 같은 프로세스 안에서 실행되므로 패치가 정상 작동합니다.


## 0.환경 설정

In [ ]:
!pip install pyhwp==0.1b15 lxml==6.0.2 'pandas==2.2.2' -q
print('설치 완료')


In [ ]:
import os, re, json, hashlib, tempfile, unicodedata
from pathlib import Path
from datetime import date
from difflib import SequenceMatcher

import pandas as pd
from lxml import etree

print('라이브러리 로드 완료')


>**pyhwp BSTR 서로게이트 패치**

`hwp5proc xml` (subprocess)로 호출하면 패치가 적용 안 됩니다. -> 터미널과 별개의 공간이기 때문


**Python API(`hwp5.xmlmodel.Hwp5File`)를 직접 호출해야** 같은 프로세스 안에서 패치가 작동합니다.


In [ ]:
import hwp5.dataio as _hwp5_dio

def _safe_decode_utf16le(data: bytes) -> str:
    try:
        return data.decode('utf-16-le')
    except UnicodeDecodeError:
        return data.decode('utf-16-le', errors='replace')

_hwp5_dio.decode_utf16le_with_hypua = _safe_decode_utf16le
print('[패치 완료] pyhwp BSTR 서로게이트 오류 무시 모드 적용')


## 1.경로 & 대상 파일 지정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR   = Path('/content/drive/MyDrive/part3data')
FILES_DIR  = BASE_DIR / 'files'
CSV_PATH   = BASE_DIR / 'data_list.csv'
OUTPUT_DIR = BASE_DIR / 'Preprocessed dataset'
DOCS_DIR   = OUTPUT_DIR / 'docs'


TARGET_DOC_ID = 'D043'

print(f'대상 doc_id : {TARGET_DOC_ID}')
out_path = DOCS_DIR / f'{TARGET_DOC_ID}.json'
print(f'출력 경로   : {out_path}')
if out_path.exists():
    print('⚠️  이미 존재합니다. 실행 시 덮어씁니다.')


**결과**

Mounted at /content/drive

대상 doc_id : D043

출력 경로   : /content/drive/MyDrive/part3data/Preprocessed dataset/docs/D043.json

## 2.CSV에서 메타데이터 읽기

In [ ]:
df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
df.columns = [
    '공고번호','공고차수','사업명','사업금액','발주기관',
    '공개일자','입찰참여시작일','입찰참여마감일',
    '사업요약','파일형식','파일명','텍스트'
]

# 기존 파이프라인과 동일한 전처리 순서
drop_mask = df['파일명'].str.contains('한국한의학연구원', na=False)
df = df[~drop_mask].reset_index(drop=True)
df['_hash'] = df['텍스트'].apply(lambda x: hashlib.md5(str(x).encode()).hexdigest())
df = df.drop_duplicates(subset='_hash', keep='first').reset_index(drop=True)
df['doc_id'] = [f'D{str(i+1).zfill(3)}' for i in range(len(df))]

UNKNOWN = '<unknown>'
for col in ['공고번호','공개일자','입찰참여시작일','입찰참여마감일','사업요약']:
    df[col] = df[col].fillna(UNKNOWN).astype(str)
df['공고차수'] = df['공고차수'].fillna(0).astype(int)
df['사업금액'] = df['사업금액'].where(df['사업금액'].notna(), other=None)

TYPE_KW = [('재구축','재구축'),('고도화','고도화'),('개선','개선'),
           ('개발','개발'),('운영','운영'),('통합','통합'),('구축','구축')]
def infer_type(name):
    for kw, lb in TYPE_KW:
        if kw in str(name): return lb
    return '기타'
df['사업유형'] = df['사업명'].apply(infer_type)
df['재공고여부'] = df['공고차수'] > 0

# NFC 정규화 파일 매핑
def nfc(s): return unicodedata.normalize('NFC', str(s).strip())
file_map = {nfc(f.name): f for f in FILES_DIR.iterdir()}
df['file_path'] = df['파일명'].apply(lambda x: file_map.get(nfc(x)) if not pd.isna(x) else None)

target_row = df[df['doc_id'] == TARGET_DOC_ID]
assert len(target_row) == 1, f'{TARGET_DOC_ID} 행을 찾지 못했습니다.'
target_row = target_row.iloc[0]

hwp_path = Path(str(target_row['file_path']))
print(f'doc_id   : {target_row["doc_id"]}')
print(f'사업명   : {target_row["사업명"]}')
print(f'파일경로 : {hwp_path}')
print(f'파일존재 : {hwp_path.exists()}')


**결과**

 doc_id   : D043

사업명   : 대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전산시스템 구축

파일경로 : /content/drive/MyDrive/part3data/files/대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp

파일존재 : True

## 3.파싱 유틸리티 정의

In [ ]:
HEADER_PATTERNS = [
    (1, re.compile(r'^[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVXivx]+[\s\.·]')),
    (1, re.compile(r'^제\s*\d+\s*장\s')),
    (2, re.compile(r'^\d+\.\s+[가-힣A-Za-z]')),
    (2, re.compile(r'^□\s')),
    (3, re.compile(r'^\s{0,4}[가나다라마바사아자차카타파하]\.\s')),
    (3, re.compile(r'^\s{0,4}\d+\)\s')),
]

def detect_header_level(line: str) -> int:
    s = line.strip()
    if len(s) > 80: return 0
    for lv, pat in HEADER_PATTERNS:
        if pat.match(s): return lv
    return 0

SKIP_CONTROL_TAGS = {'TableControl','GShapeObjectControl','EqEdit','ShapeComponent'}

def paragraph_own_text(para_el) -> str:
    texts = []
    def walk(el):
        for child in el:
            if child.tag in SKIP_CONTROL_TAGS: continue
            if child.tag == 'Text' and child.text: texts.append(child.text)
            walk(child)
    walk(para_el)
    return ' '.join(''.join(texts).split())

def cell_text(cell) -> str:
    return ' '.join(''.join(cell.itertext()).split())

def reconstruct_grid(table_el) -> list:
    rows = int(table_el.get('rows', 1))
    cols = int(table_el.get('cols', 1))
    grid = [['' for _ in range(cols)] for _ in range(rows)]
    for cell in table_el.findall('.//TableCell'):
        r, c  = int(cell.get('row',0)), int(cell.get('col',0))
        rs, cs = int(cell.get('rowspan',1)), int(cell.get('colspan',1))
        txt = cell_text(cell)
        for rr in range(r, min(r+rs, rows)):
            for cc in range(c, min(c+cs, cols)):
                grid[rr][cc] = txt
    return grid

def classify_table(grid):
    if not grid: return 'wide'
    if len(grid[0]) == 2 and len(grid) >= 2:
        first_col = [r[0] for r in grid if r[0]]
        if first_col and sum(len(v) for v in first_col)/len(first_col) < 20:
            return 'key_value'
    return 'wide'

def grid_to_markdown(grid):
    if not grid: return ''
    cols = len(grid[0])
    lines = ['| ' + ' | '.join(grid[0]) + ' |',
             '| ' + ' | '.join(['---']*cols) + ' |']
    for row in grid[1:]: lines.append('| ' + ' | '.join(row) + ' |')
    return '\n'.join(lines)

def grid_to_kv(grid):
    return ' '.join(f"{r[0]}: {r[1]}." for r in grid if len(r) >= 2 and r[0])

def serialize_table(grid):
    t = classify_table(grid)
    if t == 'key_value': return t, grid_to_kv(grid), '세로형 2열 표 → 키-값 문장'
    return t, grid_to_markdown(grid), '가로형 표 → Markdown'

NUMERIC_PATTERN = re.compile(r'^[\d,\.\-/%원₩()]+$')
def _norm(s): return re.sub(r'\s+', '', str(s))
def _tpt(grid): return ''.join(c for row in grid for c in row if c)
def _sim(a, b): return SequenceMatcher(None,a,b).ratio() if a and b else 0.0
def _rep(grid):
    cells = [_norm(c) for row in grid for c in row if c.strip()]
    return 0.0 if not cells else 1-(len(set(cells))/len(cells))
def _num(grid):
    cells = [c.strip() for row in grid for c in row if c.strip()]
    return 0.0 if not cells else sum(1 for c in cells if NUMERIC_PATTERN.match(c))/len(cells)

def classify_decorative(grid, prev):
    nr, nc = len(grid), len(grid[0]) if grid else 0
    if nr == 1 and nc == 1: return True, f'tiny_table'
    plain = _norm(_tpt(grid)); sim = 0.0
    if prev and prev.get('type')=='text':
        sim = _sim(plain, _norm(prev['content']))
        if sim >= 0.97: return True, f'text_overlap={sim:.2f}'
    rep, num = _rep(grid), _num(grid)
    if rep >= 0.5 and num < 0.1: return True, f'internal_repetition={rep:.2f}'
    return False, f'data_table(sim={sim:.2f},rep={rep:.2f},num={num:.2f})'

def _flush(sections, sc, headers, level, blocks):
    if blocks:
        sc += 1
        sid = f'S{sc:02d}'
        sections.append({
            'section_id': sid, 'header_path': list(headers), 'level': level,
            'blocks': [dict(b, block_id=f'{sid}-B{i:02d}') for i,b in enumerate(blocks,1)]
        })
    return sc

print('파싱 유틸리티 정의 완료')


## 4.HWP 파싱 함수 (Python API 직접 호출)

`hwp5.xmlmodel.Hwp5File`을 Python에서 직접 호출하여  
monkey-patch가 적용된 상태로 XML을 메모리 버퍼에 덤프합니다.


In [ ]:
import io
from hwp5.xmlmodel import Hwp5File

def parse_hwp_inprocess(hwp_path: Path) -> tuple:
    """
    pyhwp Python API로 직접 파싱.
    monkey-patch가 같은 프로세스에서 적용되므로
    HWP 5.1.x BSTR 서로게이트 오류가 무시됩니다.
    """
    # ── XML을 메모리 버퍼로 덤프 ──────────────────────────────
    buf = io.BytesIO()
    try:
        hwp = Hwp5File(str(hwp_path))
        hwp.bodytext.xmlevents().dump(buf)
    except Exception as e:
        return [], {'total_sections':0,'total_blocks':0,'text_blocks':0,
                    'table_blocks':0,'table_wide_count':0,'table_key_value_count':0,
                    'decorative_table_count':0,'decorative_table_ratio':0.0,
                    'extraction_warnings':[f'Hwp5File API 실패: {e}'],
                    'parse_method':'failed'}

    xml_bytes = buf.getvalue()
    if not xml_bytes.strip():
        return [], {'total_sections':0,'total_blocks':0,'text_blocks':0,
                    'table_blocks':0,'table_wide_count':0,'table_key_value_count':0,
                    'decorative_table_count':0,'decorative_table_ratio':0.0,
                    'extraction_warnings':['XML 버퍼 비어있음'],
                    'parse_method':'failed'}

    # ── lxml 파싱 ─────────────────────────────────────────────
    try:
        root = etree.fromstring(xml_bytes)
    except Exception as e:
        return [], {'total_sections':0,'total_blocks':0,'text_blocks':0,
                    'table_blocks':0,'table_wide_count':0,'table_key_value_count':0,
                    'decorative_table_count':0,'decorative_table_ratio':0.0,
                    'extraction_warnings':[f'lxml 파싱 실패: {e}'],
                    'parse_method':'failed'}

    # ── body_items 수집 (기존 파이프라인과 동일 로직) ──────────
    body_items = []
    for el in root.iter():
        if el.tag == 'TableBody':
            body_items.append(('table', el))
        elif el.tag == 'Paragraph':
            if 'TableCell' not in {p.tag for p in el.iterancestors()}:
                txt = paragraph_own_text(el)
                if txt: body_items.append(('text', txt))

    # ── 섹션 빌드 ─────────────────────────────────────────────
    sections, headers, level, blocks = [], ['(서두)'], 0, []
    sc = bc = tc = tbc = wc = kvc = dc = 0
    warnings = []

    for itype, ival in body_items:
        if itype == 'text':
            lv = detect_header_level(ival)
            if lv > 0:
                sc = _flush(sections, sc, headers, level, blocks); blocks = []
                if lv == 1:   headers = [ival.strip()]
                elif lv == 2: headers = headers[:1] + [ival.strip()]
                else:         headers = headers[:2] + [ival.strip()]
                level = lv
            else:
                bc += 1; tc += 1
                blocks.append({'block_id':f'B{bc:04d}','type':'text','content':ival})
        else:
            grid = reconstruct_grid(ival)
            if not grid or not any(any(c for c in r) for r in grid):
                warnings.append(f'빈 표 (blk#{bc+1})'); continue
            t_type, content, note = serialize_table(grid)
            if not content.strip():
                warnings.append(f'표 직렬화 빈 결과 (blk#{bc+1})'); continue
            bc += 1; tbc += 1
            wc += t_type == 'wide'; kvc += t_type == 'key_value'
            prev = blocks[-1] if blocks else None
            is_dec, dec_r = classify_decorative(grid, prev)
            if is_dec: dc += 1
            blocks.append({'block_id':f'B{bc:04d}','type':'table','table_type':t_type,
                           'content':content,'note':note,'raw_grid':grid,
                           'is_decorative':is_dec,'decorative_reason':dec_r})

    _flush(sections, sc, headers, level, blocks)

    qa = {'total_sections':len(sections),'total_blocks':bc,
          'text_blocks':tc,'table_blocks':tbc,
          'table_wide_count':wc,'table_key_value_count':kvc,
          'decorative_table_count':dc,
          'decorative_table_ratio':round(dc/tbc,3) if tbc else 0.0,
          'extraction_warnings':warnings,
          'parse_method':'A1_inprocess_api'}
    return sections, qa

print('parse_hwp_inprocess() 정의 완료')
print('방식: hwp5.xmlmodel.Hwp5File → BytesIO 버퍼 → lxml 파싱')


## 5.JSON 빌드 함수

In [ ]:
TOC_PATTERN = re.compile(
    r'^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVXivx]+[^\n]{2,40}?|\d+\.\s[가-힣A-Za-z][^\n]{2,40}?)'
    r'[\s·\-·⋯\.]{3,}\s*\d+', re.MULTILINE)

def extract_toc(text): return [m.strip() for m in TOC_PATTERN.findall(str(text))][:30]

def compute_toc_match_rate(toc, sections):
    if not toc: return 0.0
    all_headers = ' '.join(h for s in sections for h in s['header_path'])
    return round(sum(1 for t in toc if t[:6] in all_headers)/len(toc), 2)

def build_json(row, sections, qa, toc):
    all_text = ' '.join(b['content'] for s in sections for b in s['blocks'])
    qa['toc_header_match_rate'] = compute_toc_match_rate(toc, sections)
    qa['dedup_hash'] = 'sha256:' + hashlib.sha256(all_text.encode()).hexdigest()
    if qa.get('decorative_table_ratio',0) > 0.15:
        qa['extraction_warnings'].append(f'high_decorative_table_ratio: {qa["decorative_table_ratio"]:.1%}')
    return {
        'schema_version':'1.0',
        'doc_id': row['doc_id'], 'file_name': str(row['파일명']),
        'source_format': str(row['파일형식']).lower(), 'processed_at': str(date.today()),
        'metadata': {
            '공고번호': row['공고번호'], '공고차수': int(row['공고차수']),
            '사업명': row['사업명'],
            '사업금액': int(row['사업금액']) if pd.notna(row['사업금액']) else None,
            '발주기관': row['발주기관'], '공개일자': row['공개일자'],
            '입찰참여시작일': row['입찰참여시작일'], '입찰참여마감일': row['입찰참여마감일'],
            '사업요약': row['사업요약'], '사업유형': row['사업유형'],
            '재공고여부': bool(row['재공고여부']), 'linked_doc_id': None,
            '목차존재': len(toc) > 0
        },
        'toc': toc, 'sections': sections, 'qa': qa
    }

print('JSON 빌드 함수 정의 완료')


## 6.파싱 & 저장

In [ ]:
print(f'=== {TARGET_DOC_ID} 파싱 시작 ===')
print(f'파일: {hwp_path.name}')
print()

assert hwp_path.exists(), f'파일 없음: {hwp_path}\nFILES_DIR를 확인하세요: {FILES_DIR}'

sections, qa_info = parse_hwp_inprocess(hwp_path)

if qa_info['total_sections'] == 0:
    reason = qa_info['extraction_warnings'][0] if qa_info['extraction_warnings'] else '알 수 없는 실패'
    print(f'❌ 파싱 실패: {reason}')
else:
    toc = extract_toc(target_row.get('텍스트', ''))
    doc_json = build_json(target_row, sections, qa_info, toc)

    out_path = DOCS_DIR / f'{TARGET_DOC_ID}.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(doc_json, f, ensure_ascii=False, indent=2)

    print(f'✅ 저장 완료 : {out_path}')
    print(f'parse_method : {qa_info["parse_method"]}')
    print(f'섹션 수      : {qa_info["total_sections"]}')
    print(f'전체 블록    : {qa_info["total_blocks"]}')
    print(f'표 블록      : {qa_info["table_blocks"]}')
    print(f'경고         : {qa_info["extraction_warnings"]}')
    print()
    print('docs/ 폴더의 다른 파일은 변경되지 않았습니다.')


**결과**

저장 완료 : /content/drive/MyDrive/part3data/Preprocessed dataset/docs/D043.json

parse_method : A1_inprocess_api

섹션 수      : 55

전체 블록    : 696

표 블록      : 113

경고         : ['high_decorative_table_ratio: 16.8%']

docs/ 폴더의 다른 파일은 변경되지 않았습니다.

## 7.결과 확인

In [ ]:
out_path = DOCS_DIR / f'{TARGET_DOC_ID}.json'
with open(out_path, encoding='utf-8') as f:
    doc = json.load(f)

print(f'doc_id       : {doc["doc_id"]}')
print(f'사업명       : {doc["metadata"]["사업명"]}')
print(f'parse_method : {doc["qa"]["parse_method"]}')
print(f'섹션 수      : {doc["qa"]["total_sections"]}')
print(f'표 블록      : {doc["qa"]["table_blocks"]}')
print()
print('── 섹션 앞 5개 ──')
for s in doc['sections'][:5]:
    print(f'  [{s["section_id"]}] {s["header_path"]} — {len(s["blocks"])}블록')
print()
print('── 표 있는 섹션 앞 2개 ──')
for s in [s for s in doc['sections'] if any(b["type"]=="table" for b in s["blocks"])][:2]:
    print(f'  [{s["section_id"]}] {s["header_path"]}')
    for b in [b for b in s["blocks"] if b["type"]=="table"][:1]:
        print(f'    {b["content"][:100].replace(chr(10)," ")}')


doc_id       : D043

사업명       : 대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전산시스템 구축

parse_method : A1_inprocess_api

섹션 수      : 55

표 블록      : 113



── 섹션 앞 5개 ──

  [S01] ['(서두)'] — 7블록

  [S02] ['Ⅳ. 제안서 제출안내 31', '2. 제안 평가(PT) 32'] — 72블록

  [S03] ['Ⅳ. 제안서 제출안내 31', '1. 기능 요구사항 (System Function Requirement)'] — 8블록
  
  [S04] ['Ⅳ. 제안서 제출안내 31', '2. 성능 요구사항(PER : Performance Funtion Requirement)'] — 4블록
  
  [S05] ['Ⅳ. 제안서 제출안내 31', '3. 데이터 요구사항(DAR : Data Requirement)'] — 2블록


── 표 있는 섹션 앞 2개 ──
  
  [S01] ['(서두)']
    |  | | --- | | (재공고)대전대학교 다층적 융합 학습경험플랫폼(MILE) 구축 제안요청서 | |  |
  
  [S02] ['Ⅳ. 제안서 제출안내 31', '2. 제안 평가(PT) 32']
    | 1 |  | 사업개요 | | --- | --- | --- |

# [Phase 2] D043 목차 병합 및 구조 후처리
**D043 전용 — toc 병합 + 후처리 → final_data 저장**

기존 Phase 2 파이프라인에서 제외됐던 D043 1건만 처리합니다.  
나머지 `final_data/` 파일은 전혀 건드리지 않습니다.

## 전제 조건
- Phase 1 단일 파일 파이프라인으로 `docs/D043.json` 이 생성되어 있어야 합니다.
- `toc_docs/D043_toc.json` 이 존재해야 합니다.

## 처리 흐름
```
docs/D043.json          ← Phase 1 결과
toc_docs/D043_toc.json  ← 사람이 만든 목차
        │
        ▼
[Step 1] toc 병합          — toc 필드를 사람이 만든 목차로 교체
[Step 2] header_path ↔ toc 매칭 — toc_ref 부여 (SequenceMatcher, threshold=0.45)
[Step 3] 표 병합 셀 중복 정리  — 같은 행 내 반복 셀 제거
[Step 4] 대형 섹션 분할 표시   — block_count > 30 이면 needs_subsplit=True
[Step 5] qa 재계산 & 저장     — final_data/D043.json
```


In [ ]:
import json, re
from pathlib import Path
from difflib import SequenceMatcher
from typing import Any
import pandas as pd

print('라이브러리 로드 완료')


## 1.경로 & 파일 로드

> `TARGET_DOC_ID`는 `'D043'` 고정입니다.  
> 경로는 기존 Phase 2 파이프라인과 완전히 동일합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR   = Path('/content/drive/MyDrive/part3data')
OUTPUT_DIR = BASE_DIR / 'Preprocessed dataset'
DOCS_DIR   = OUTPUT_DIR / 'docs'       # Phase 1 결과
TOC_DIR    = OUTPUT_DIR / 'toc_docs'   # 사람이 만든 목차
FINAL_DIR  = OUTPUT_DIR / 'final_data' # 최종 출력

FINAL_DIR.mkdir(parents=True, exist_ok=True)

TARGET_DOC_ID = 'D043'  # 고정

doc_path = DOCS_DIR  / f'{TARGET_DOC_ID}.json'
toc_path = TOC_DIR   / f'{TARGET_DOC_ID}_toc.json'
out_path = FINAL_DIR / f'{TARGET_DOC_ID}.json'

print(f'입력 doc  : {doc_path}  → 존재: {doc_path.exists()}')
print(f'입력 toc  : {toc_path}  → 존재: {toc_path.exists()}')
print(f'출력 경로 : {out_path}')
if out_path.exists():
    print('⚠️  출력 파일이 이미 존재합니다. 실행 시 덮어씁니다.')

assert doc_path.exists(), f'❌ {doc_path} 없음 — Phase 1 단일 파이프라인을 먼저 실행하세요.'
assert toc_path.exists(), f'❌ {toc_path} 없음 — toc_docs 폴더에 D043_toc.json을 준비하세요.'


**결과**

Mounted at /content/drive

입력 doc  : /content/drive/MyDrive/part3data/Preprocessed dataset/docs/D043.json  → 존재: True

입력 toc  : /content/drive/MyDrive/part3data/Preprocessed dataset/toc_docs/D043_toc.json  → 존재: True

출력 경로 : /content/drive/MyDrive/part3data/Preprocessed dataset/final_data/D043.json

In [ ]:
# 두 파일 로드
with open(doc_path, encoding='utf-8') as f:
    main_data = json.load(f)

with open(toc_path, encoding='utf-8') as f:
    toc_data = json.load(f)

# doc_id 이중 확인
assert main_data['doc_id'] == TARGET_DOC_ID,     f'doc JSON 내부 doc_id 불일치: {main_data["doc_id"]} ≠ {TARGET_DOC_ID}'
assert toc_data['doc_id'] == TARGET_DOC_ID,     f'toc JSON 내부 doc_id 불일치: {toc_data["doc_id"]} ≠ {TARGET_DOC_ID}'

print(f'doc_id   : {main_data["doc_id"]}')
print(f'사업명   : {main_data["metadata"]["사업명"]}')
print(f'섹션 수  : {len(main_data["sections"])}')
print(f'toc 항목 : {len(toc_data["toc"])}개')
print()
print('toc 상위 5개:')
for t in toc_data['toc'][:5]:
    print(' ', t)


**결과**
doc_id   : D043

사업명   : 대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전산시스템 구축

섹션 수  : 55

toc 항목 : 75개


toc 상위 5개:

  {'order': 1, 'level': 1, 'number': 'Ⅰ', 'title': '사업 안내', 'page': 4, 'is_uncertain': False}

  {'order': 2, 'level': 2, 'number': '1.', 'title': '사업개요', 'page': 4, 'is_uncertain': False}

  {'order': 3, 'level': 2, 'number': '2.', 'title': '사업배경 및 목적', 'page': 4, 'is_uncertain': False}

  {'order': 4, 'level': 2, 'number': '3.', 'title': '사업 기대효과', 'page': 7, 'is_uncertain': False}

  {'order': 5, 'level': 1, 'number': 'Ⅱ', 'title': '제안 요청사항', 'page': 8, 'is_uncertain': False}


## 2.toc 병합

`docs/D043.json`의 `toc` 필드(Phase 1에서 자동 추출, 부실)를  
`toc_docs/D043_toc.json`의 사람이 만든 목차로 교체합니다.


In [ ]:
# toc 교체
before_toc_len = len(main_data.get('toc', []))
main_data['toc'] = toc_data['toc']
after_toc_len  = len(main_data['toc'])

uncertain_cnt = sum(1 for t in toc_data['toc'] if t.get('is_uncertain'))

print(f'Step 1 완료 — toc 병합')
print(f'  교체 전 toc 항목 수 : {before_toc_len}')
print(f'  교체 후 toc 항목 수 : {after_toc_len}')
print(f'  is_uncertain 항목   : {uncertain_cnt}건 {"⚠️ 검수 권장" if uncertain_cnt else ""}')


**결과**

Step 1 완료 — toc 병합

  교체 전 toc 항목 수 : 4

  교체 후 toc 항목 수 : 75
  
  is_uncertain 항목   : 1건 ⚠️ 검수 권장

## 3.header_path ↔ toc 매칭

각 섹션의 `header_path` 마지막 텍스트와 toc 항목 title을  
`SequenceMatcher`로 비교해 가장 유사한 항목을 `toc_ref`로 연결합니다.

- threshold = 0.45 (기존 Phase 2와 동일)
- 매칭 실패 섹션은 `toc_ref: null`, `toc_match_failed: true`


In [ ]:
def similarity(a: Any, b: Any) -> float:
    a_s = str(a) if a is not None else ''
    b_s = str(b) if b is not None else ''
    return SequenceMatcher(None, a_s, b_s).ratio()

def match_section_to_toc(header_text: str, toc_items: list, threshold: float = 0.45):
    best, best_score = None, 0.0
    for item in toc_items:
        toc_title = item.get('title', '') if isinstance(item, dict) else ''
        score = similarity(header_text, toc_title)
        if score > best_score:
            best, best_score = item, score
    if best is not None and best_score >= threshold:
        return best, best_score
    return None, best_score

def attach_toc_ref(main_data: dict, threshold: float = 0.45) -> list:
    failed = []
    toc_items = main_data.get('toc', [])
    for section in main_data['sections']:
        header_text = str(section['header_path'][-1]) if section.get('header_path') else ''
        matched, score = match_section_to_toc(header_text, toc_items, threshold)
        if matched:
            section['toc_ref'] = {
                'order': matched['order'],
                'number': matched['number'],
                'title': matched['title'],
                'match_score': round(score, 3),
            }
            section['toc_match_failed'] = False
        else:
            section['toc_ref'] = None
            section['toc_match_failed'] = True
            failed.append({
                'section_id': section['section_id'],
                'header_text': header_text,
                'best_score': round(score, 3),
            })
    return failed

match_failures = attach_toc_ref(main_data, threshold=0.45)

matched_cnt = sum(1 for s in main_data['sections'] if not s['toc_match_failed'])
total_cnt   = len(main_data['sections'])

print(f'Step 2 완료 — toc 매칭')
print(f'  전체 섹션     : {total_cnt}개')
print(f'  매칭 성공     : {matched_cnt}개')
print(f'  매칭 실패     : {len(match_failures)}개')
if match_failures:
    print()
    print('  매칭 실패 섹션 목록:')
    for row in match_failures:
        print(f'    [{row["section_id"]}] {row["header_text"]!r:30s}  best_score={row["best_score"]}')


Step 2 완료 — toc 매칭

  전체 섹션     : 55개

  매칭 성공     : 27개

  매칭 실패     : 28개
  

## 4.표 병합 셀 중복 정리

같은 행 안에서 바로 앞 칸과 동일한 값이 연속되면  
병합 셀 흔적으로 판단하고 제거합니다.  
정리 후 1행 1열로 줄어들면 장식용 표(`is_decorative=True`)로 재분류합니다.


In [ ]:
def dedup_merged_cells(raw_grid: list) -> list:
    cleaned = []
    for row in raw_grid:
        new_row, prev = [], object()
        for cell in row:
            if cell == prev:
                continue
            new_row.append(cell)
            prev = cell
        cleaned.append(new_row)
    return cleaned

def grid_to_markdown(grid: list) -> str:
    if not grid: return ''
    rows = ['| ' + ' | '.join(str(c) for c in row) + ' |' for row in grid]
    sep  = '| ' + ' | '.join(['---'] * len(grid[0])) + ' |'
    return '\n'.join([rows[0], sep] + rows[1:]) if len(rows) > 1 else rows[0] + '\n' + sep

def clean_tables(main_data: dict) -> int:
    changed = 0
    for section in main_data['sections']:
        for block in section['blocks']:
            if block.get('type') != 'table':
                continue
            before = block.get('raw_grid', [])
            after  = dedup_merged_cells(before)
            if after != before:
                block['raw_grid'] = after
                block['content']  = grid_to_markdown(after)
                changed += 1
            flat = [c for row in block['raw_grid'] for c in row]
            if len(block['raw_grid']) <= 1 and len(flat) <= 1 and not block.get('is_decorative'):
                block['is_decorative']   = True
                block['decorative_reason'] = 'dedup_collapsed_to_single_cell'
    return changed

tables_cleaned = clean_tables(main_data)
print(f'Step 3 완료 — 표 병합 셀 중복 정리')
print(f'  변경된 표 블록 수 : {tables_cleaned}개')


Step 3 완료 — 표 병합 셀 중복 정리

  변경된 표 블록 수 : 58개

## 5.대형 섹션 분할 표시

블록 수가 30개를 초과하는 섹션에 `needs_subsplit=True` 플래그를 부여합니다.  
청킹 단계에서 해당 섹션을 우선 검토할 수 있도록 신호만 남깁니다.


In [ ]:
BLOCK_THRESHOLD = 30

def flag_large_sections(main_data: dict, threshold: int = BLOCK_THRESHOLD) -> list:
    flagged = []
    for section in main_data['sections']:
        block_count = len(section['blocks'])
        section['block_count']   = block_count
        section['needs_subsplit'] = block_count > threshold
        if section['needs_subsplit']:
            flagged.append({
                'section_id':  section['section_id'],
                'block_count': block_count,
                'header_path': section['header_path'],
                'toc_ref':     section.get('toc_ref'),
            })
    return flagged

large_sections = flag_large_sections(main_data)

print(f'Step 4 완료 — 대형 섹션 분할 표시 (기준: block_count > {BLOCK_THRESHOLD})')
print(f'  분할 필요 섹션 : {len(large_sections)}개')
if large_sections:
    print()
    for row in sorted(large_sections, key=lambda r: -r['block_count']):
        print(f'  [{row["section_id"]}] block_count={row["block_count"]}  {row["header_path"]}')


**결과**

Step 4 완료 — 대형 섹션 분할 표시 (기준: block_count > 30)

  분할 필요 섹션 : 5개


  [S28] block_count=136  ['Ⅳ. 제안서 제출안내 31', '10. 사업완료 후 무상유지보수 기간 보안관리', '나. 대학이 제시하는 제안사항과 업체선정과정 및 평가 기준에 대하여 100% 동의하는 자']

  [S55] block_count=91  ['Ⅳ. 제안서 제출안내 31', '2. 제출방법 및 양식', '나. 위 기록 중 다음과 같은 분석자료 제출']

  [S02] block_count=72  ['Ⅳ. 제안서 제출안내 31', '2. 제안 평가(PT) 32']

  [S39] block_count=56  ['Ⅳ. 제안서 제출안내 31', '5. 최신 계약서 서식', '나. 계약서 작성 전에 [포털-통합정보-공통-개인정보보호안내-관련자료]에서 다운로드 받아 사용하시기 바랍니다.']

  [S40] block_count=45  ['Ⅳ. 제안서 제출안내 31', '2. 용역사업 보안위규 처리기준']


## 6.qa 재계산 & final_data 저장

Step 1~4로 변경된 내용을 반영해 qa 통계를 재계산하고  
`final_data/D043.json`으로 저장합니다.  
`docs/D043.json` 원본은 건드리지 않습니다.


In [ ]:
def recompute_qa(main_data: dict) -> dict:
    sections     = main_data['sections']
    total_blocks = sum(len(s['blocks']) for s in sections)
    table_blocks = sum(1 for s in sections for b in s['blocks'] if b['type'] == 'table')
    text_blocks  = total_blocks - table_blocks
    decorative   = sum(1 for s in sections for b in s['blocks'] if b.get('is_decorative'))
    matched      = sum(1 for s in sections if not s.get('toc_match_failed'))
    needs_split  = sum(1 for s in sections if s.get('needs_subsplit'))

    qa = main_data.get('qa', {})
    qa.update({
        'total_sections':        len(sections),
        'total_blocks':          total_blocks,
        'text_blocks':           text_blocks,
        'table_blocks':          table_blocks,
        'decorative_table_count': decorative,
        'decorative_table_ratio': round(decorative / table_blocks, 3) if table_blocks else 0.0,
        'toc_header_match_rate': round(matched / len(sections), 3) if sections else 0.0,
        'needs_subsplit_count':  needs_split,
    })
    main_data['qa'] = qa
    return qa

qa = recompute_qa(main_data)

with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(main_data, f, ensure_ascii=False, indent=2)

print(f'Step 5 완료 — qa 재계산 & 저장')
print(f'  저장 경로            : {out_path}')
print(f'  total_sections       : {qa["total_sections"]}')
print(f'  total_blocks         : {qa["total_blocks"]}')
print(f'  table_blocks         : {qa["table_blocks"]}')
print(f'  toc_header_match_rate: {qa["toc_header_match_rate"]}')
print(f'  needs_subsplit_count : {qa["needs_subsplit_count"]}')
print()
print('docs/D043.json 원본은 변경되지 않았습니다.')


Step 5 완료 — qa 재계산 & 저장

  저장 경로            : /content/drive/MyDrive/part3data/Preprocessed dataset/final_data/D043.json

  total_sections       : 55

  total_blocks         : 696

  table_blocks         : 113

  toc_header_match_rate: 0.491

  needs_subsplit_count : 5
  

docs/D043.json 원본은 변경되지 않았습니다.

## 7.최종 검증

In [ ]:
with open(out_path, encoding='utf-8') as f:
    check = json.load(f)

print(f'doc_id               : {check["doc_id"]}')
print(f'사업명               : {check["metadata"]["사업명"]}')
print(f'toc 항목 수          : {len(check["toc"])}')
print(f'toc_header_match_rate: {check["qa"]["toc_header_match_rate"]}')
print(f'needs_subsplit_count : {check["qa"]["needs_subsplit_count"]}')
print()
print('── toc 상위 5개 ──')
for t in check['toc'][:5]:
    print(' ', t)
print()
print('── 섹션별 toc_ref 상태 (앞 8개) ──')
for s in check['sections'][:8]:
    ref   = s.get('toc_ref')
    ref_s = f'{ref["number"]} {ref["title"]} (score={ref["match_score"]})' if ref else 'None'
    split = '⚠️ subsplit' if s.get('needs_subsplit') else ''
    print(f'  [{s["section_id"]}] {str(s["header_path"])[:40]:42s} → {ref_s} {split}')


결과

doc_id               : D043

사업명               : 대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전산시스템 구축

toc 항목 수          : 75

toc_header_match_rate: 0.491

needs_subsplit_count : 5


── toc 상위 5개 ──

  {'order': 1, 'level': 1, 'number': 'Ⅰ', 'title': '사업 안내', 'page': 4, 'is_uncertain': False}

  {'order': 2, 'level': 2, 'number': '1.', 'title': '사업개요', 'page': 4, 'is_uncertain': False}

  {'order': 3, 'level': 2, 'number': '2.', 'title': '사업배경 및 목적', 'page': 4, 'is_uncertain': False}

  {'order': 4, 'level': 2, 'number': '3.', 'title': '사업 기대효과', 'page': 7, 'is_uncertain': False}

  {'order': 5, 'level': 1, 'number': 'Ⅱ', 'title': '제안 요청사항', 'page': 8, 'is_uncertain': False}


── 섹션별 toc_ref 상태 (앞 8개) ──

  [S01] ['(서두)']                                   → None
  [S02] ['Ⅳ. 제안서 제출안내 31', '2. 제안 평가(PT) 32']      → 2. 제안 평가(PT) (score=0.75) ⚠️ subsplit

  [S03] ['Ⅳ. 제안서 제출안내 31', '1. 기능 요구사항 (System F   → 1. 기능 요구사항 (System Function Requirement) (score=0.961)

  [S04] ['Ⅳ. 제안서 제출안내 31', '2. 성능 요구사항(PER : Per   → 2. 성능 요구사항(PER : Performance Funtion Requirement) (score=0.968)

  [S05] ['Ⅳ. 제안서 제출안내 31', '3. 데이터 요구사항(DAR : Da   → 3. 데이터 요구사항(DAR : Data Requirement) (score=0.955)

  [S06] ['Ⅳ. 제안서 제출안내 31', '4. 테스트 요구사항(TER : Te   → 4. 테스트 요구사항(TER : Test Requirement) (score=0.955)

  [S07] ['Ⅳ. 제안서 제출안내 31', '5. 품질 및 하자보증']         → 5. 품질 및 하자보증 (score=0.857)
  
  [S08] ['Ⅳ. 제안서 제출안내 31', '6. 웹 접근성 준수']          → 6. 웹 접근성 준수 (score=0.842)